# Lens Distortion Correction — Google Colab Training

**Before starting:**
1. Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro+)
2. Upload your dataset zip to Google Drive (do this once)
3. Upload this project's code folder to Drive as well (or use the git clone cell)

**Handling session timeouts:** Checkpoints are saved to Drive after every epoch.
If your session dies, just re-run all cells — training will resume automatically.

## 1. Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
!nvidia-smi -L

In [ ]:
# Install extra deps (torch/torchvision are pre-installed on Colab)
!pip install -q timm kornia albumentations scikit-image

## 2. Configure Paths

Edit the Drive paths below to match where you uploaded your files.

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

# ── Edit these to match your Drive layout ─────────────────────────────────────
DRIVE_ROOT        = Path('/content/drive/MyDrive/lens-correction')
DRIVE_DATASET_ZIP = DRIVE_ROOT / 'automatic-lens-correction.zip'  # your uploaded zip
DRIVE_CODE_DIR    = DRIVE_ROOT / 'code'                           # your uploaded project folder
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'                    # persists across sessions
DRIVE_OUTPUTS     = DRIVE_ROOT / 'outputs'                        # submission zip saved here
# ──────────────────────────────────────────────────────────────────────────────

# Fast local scratch (lost on session end, but Drive has the checkpoints)
LOCAL_ROOT     = Path('/content/lens-correction')
LOCAL_DATA     = LOCAL_ROOT / 'data_raw'
LOCAL_CODE     = LOCAL_ROOT / 'code'
LOCAL_CKPTS    = LOCAL_ROOT / 'checkpoints'
LOCAL_OUTPUTS  = LOCAL_ROOT / 'outputs'

for p in [LOCAL_DATA, LOCAL_CODE, LOCAL_CKPTS, LOCAL_OUTPUTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUTS]:
    p.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Dataset zip exists:', DRIVE_DATASET_ZIP.exists())
print('Code dir exists:', DRIVE_CODE_DIR.exists())

## 3. Set Up Code

**Option A** — copy from Drive (if you uploaded the project folder there)  
**Option B** — git clone from GitHub (if you pushed your code there)

In [ ]:
# Option A: copy from Drive
if DRIVE_CODE_DIR.exists():
    !cp -r {DRIVE_CODE_DIR}/* {LOCAL_CODE}/
    print('Code copied from Drive.')

# Option B: git clone (uncomment and fill in your repo URL)
# !git clone https://github.com/YOUR_USERNAME/baba-camera-correct.git {LOCAL_CODE}

sys.path.insert(0, str(LOCAL_CODE))
print('Python path set to:', LOCAL_CODE)

## 4. Extract Dataset (once per session)

The zip lives on Drive; we extract to local `/content` for fast I/O during training.
This takes ~5 minutes and only needs to run once per session.

In [ ]:
import time

TRAIN_EXTRACTED = LOCAL_DATA / 'lens-correction-train-clea'
TEST_EXTRACTED  = LOCAL_DATA / 'test-originals'

if not TRAIN_EXTRACTED.exists():
    print(f'Extracting dataset from {DRIVE_DATASET_ZIP} ...')
    t0 = time.time()
    with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    print(f'Done in {time.time()-t0:.0f}s')
else:
    print('Dataset already extracted, skipping.')

train_samples = len(list(TRAIN_EXTRACTED.iterdir())) if TRAIN_EXTRACTED.exists() else 0
test_images   = len(list(TEST_EXTRACTED.glob('*.jpg'))) if TEST_EXTRACTED.exists() else 0
print(f'Train samples: {train_samples}')
print(f'Test images:   {test_images}')

## 5. Configure Paths in config.py

In [ ]:
import config

config.TRAIN_DIR       = TRAIN_EXTRACTED
config.TEST_DIR        = TEST_EXTRACTED
config.OUTPUT_DIR      = LOCAL_OUTPUTS
config.CHECKPOINT_DIR  = LOCAL_CKPTS     # local during training
config.SUBMISSION_DIR  = LOCAL_OUTPUTS / 'submissions'
config.SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# Colab T4 has 15 GB VRAM — reduce batch sizes vs the default
config.PHASE1.batch_size = 6   # default 8
config.PHASE2.batch_size = 2   # default 4
config.PHASE2.resolution = 640  # default 768 — lower if OOM; raise for A100

print('PHASE1:', config.PHASE1)
print('PHASE2:', config.PHASE2)

## 6. Resume Helper

Copies the latest checkpoint from Drive → local before training starts,
and copies back after every epoch so progress is never lost to a session crash.

In [ ]:
def sync_checkpoints_from_drive():
    """Copy any Drive checkpoints to local before training."""
    ckpts = list(DRIVE_CHECKPOINTS.glob('*.pt'))
    if ckpts:
        for f in ckpts:
            dest = LOCAL_CKPTS / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                print(f'  Restored from Drive: {f.name}')
    else:
        print('  No Drive checkpoints found — starting fresh.')

def sync_checkpoints_to_drive():
    """Copy all local checkpoints to Drive (called after each epoch in a hook)."""
    for f in LOCAL_CKPTS.glob('*.pt'):
        dest = DRIVE_CHECKPOINTS / f.name
        shutil.copy2(f, dest)

sync_checkpoints_from_drive()

## 7. Patch Training Loop to Sync Checkpoints After Each Save

We monkey-patch `save_checkpoint` to also copy to Drive immediately.

In [ ]:
import training.train as train_module

_original_save = train_module.save_checkpoint

def _save_and_sync(*args, **kwargs):
    _original_save(*args, **kwargs)
    sync_checkpoints_to_drive()
    print('  Synced checkpoints to Drive.')

train_module.save_checkpoint = _save_and_sync
print('Checkpoint sync hook installed.')

## 8. Phase 1 — Parametric Only

Auto-resumes from the best Phase 1 checkpoint if one exists.

In [ ]:
import torch
from training.train import train_phase

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# Auto-resume if a Phase 1 checkpoint exists
p1_best = LOCAL_CKPTS / 'phase1_parametric_best.pt'
if p1_best.exists():
    config.PHASE1.resume_from = str(p1_best)
    print(f'Resuming Phase 1 from {p1_best}')
else:
    print('Starting Phase 1 from scratch.')

model = train_phase(config.PHASE1, device)

## 9. Phase 2 — Full Pipeline

Auto-resumes from best Phase 2 checkpoint, or starts from Phase 1 weights.

In [ ]:
p2_best = LOCAL_CKPTS / 'phase2_full_best.pt'
if p2_best.exists():
    config.PHASE2.resume_from = str(p2_best)
    print(f'Resuming Phase 2 from {p2_best}')
    model = None  # will be loaded from checkpoint inside train_phase
else:
    print('Starting Phase 2 from Phase 1 weights.')
    # model already holds Phase 1 weights from cell above

model = train_phase(config.PHASE2, device, model=model)

## 10. Quick Validation

In [ ]:
from torch.utils.data import DataLoader
from data.dataset import LensTrainDataset
from losses.combined import CombinedLoss
from training.validate import run_validation

val_ds = LensTrainDataset(resolution=config.PHASE2.resolution, split='val', augment=False)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
criterion = CombinedLoss(config.PHASE2_LOSS)

val_results = run_validation(
    model, val_loader, criterion, device,
    compute_proxy=True, max_proxy_samples=50,
)
print('\nValidation:')
for k, v in sorted(val_results.items()):
    print(f'  {k:30s}: {v:.4f}')

## 11. Inference on Test Images

In [ ]:
from inference.predict import run_inference

run_inference(
    checkpoint_path=str(LOCAL_CKPTS / 'phase2_full_best.pt'),
    output_dir=str(config.SUBMISSION_DIR / 'final'),
    resolution=config.PHASE2.resolution,
    do_refine=True,
    batch_size=4,
)

## 12. Package & Save Submission Zip to Drive

In [ ]:
sub_dir  = config.SUBMISSION_DIR / 'final'
zip_local = config.SUBMISSION_DIR / 'submission.zip'
zip_drive = DRIVE_OUTPUTS / 'submission.zip'

files = sorted(sub_dir.glob('*.jpg'))
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_STORED) as zf:
    for f in files:
        zf.write(f, f.name)

# Copy to Drive so it survives session end
shutil.copy2(zip_local, zip_drive)

print(f'submission.zip: {zip_local.stat().st_size/1e6:.1f} MB, {len(files)} images')
print(f'Saved to Drive: {zip_drive}')
print('\nNext: upload submission.zip to bounty.autohdr.com')

## 13. Preview Sample Outputs

In [ ]:
import matplotlib.pyplot as plt
import cv2, numpy as np

sample_files = sorted(sub_dir.glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, f in zip(axes.flat, sample_files):
    img = cv2.imread(str(f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f.stem[:24], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Corrected Outputs', fontsize=14)
plt.tight_layout()
plt.show()